# API hardcover для получения рейтинга книги

In [67]:
import pandas as pd
import requests
df = pd.read_excel('nyt_books.xlsx').head(5)

df.columns

Index(['Unnamed: 0', 'age_group', 'amazon_product_url', 'article_chapter_link',
       'asterisk', 'author', 'book_image', 'book_image_height',
       'book_image_width', 'book_review_link', 'book_uri', 'contributor',
       'contributor_note', 'created_date', 'dagger', 'description',
       'first_chapter_link', 'price', 'primary_isbn10', 'primary_isbn13',
       'publisher', 'rank', 'rank_last_week', 'sunday_review_link', 'title',
       'updated_date', 'weeks_on_list', 'isbns', 'buy_links'],
      dtype='str')

In [68]:
from dotenv import load_dotenv

load_dotenv()

True

In [69]:
import os

HARDCOVER_API_KEY = os.environ.get("HARDCOVER_API_KEY")
HARDCOVER_URL = "https://api.hardcover.app/v1/graphql"

Установим паузу между запросами, так как всего разрешено 60 запросов в минуту по правилам данного апи

In [70]:
pause = 1

headers = {
    "Authorization": f"Bearer {HARDCOVER_API_KEY}",
    "Content-Type": "application/json"
}


In [72]:
import time

def search_book(title, author):
    query = """
        query($q: String!) {
            search(query: $q, query_type: "Book", per_page: 5) {
                results
            }
        }
        """
    response = requests.post(
        HARDCOVER_URL,
        headers=headers,
        json={
            "query": query,
            "variables": {
                "q": f"{title} {author}"
            }
        }
    )

    data = response.json()

    if data is None:
        return None
       
    return data['data']


Книги, найденные по запросу название + автор

In [79]:
search_book("THE DRAGON'S APPRENTICE", "Delia Owens")

{'search': {'results': {'facet_counts': [],
   'found': 6,
   'hits': [{'document': {'activities_count': 12,
      'alternative_titles': ["The Dragon's Apprentice"],
      'author_names': ['James A. Owen'],
      'compilation': False,
      'content_warnings': [],
      'contribution_types': ['Author'],
      'contributions': [{'author': {'id': 157121,
         'image': {'color': '#ab8f69',
          'color_name': 'Gray',
          'height': 265,
          'id': 5690462,
          'url': 'https://assets.hardcover.app/author/157121/d5b4aac0-8f9c-4816-9f0b-9e97bb7c97ea.png',
          'width': 265},
         'name': 'James A. Owen',
         'slug': 'james-a-owen'}}],
      'cover_color': 'Green',
      'featured_series': {'details': '5',
       'featured': False,
       'id': 73585,
       'position': 5.0,
       'series': {'books_count': 15,
        'id': 6106,
        'name': 'The Chronicles of the Imaginarium Geographica',
        'primary_books_count': 8,
        'slug': 'the-chroni

Для поиска лучшего результата из тех, которые мы получили в responce, будем использовать функцию для сравнения авторов

In [80]:
def norm_author(author):
    author_norm = author.lower().strip()
    return " ".join(author_norm.split())

In [93]:
def find_first_book_data(title, author):
    correct_author = norm_author(author)
    
    data = search_book(title, author)

    if data is None:
        return None, None, None, None, None, None, None

    results = data["search"].get("results", []).get("hits", None)

    if results is None:
        return None, None, None, None, None, None, None
    

    for hit in results:
        item = hit.get("document", hit)
        authors = item.get("author_names", [])

        for name in authors:
            name = norm_author(name)
            if name and ((name in correct_author) or (correct_author in name)):
                return item.get("title"), item.get("author_names"), item.get("rating"), item.get("ratings_count"), item.get("release_year"), item.get("users_count"), item.get("genres")


    return None, None, None, None, None, None, None



In [94]:
find_first_book_data("The Last Wish", "Andrzej Sapkowski")

('The Last Wish',
 ['Andrzej Sapkowski', 'Danusia Stok'],
 4.055982436882546,
 1822,
 1975,
 3532,
 ['Fantasy',
  'Adventure',
  'Science Fiction & Fantasy',
  'Young Adult',
  'War',
  'Fantasy fiction',
  'Historical',
  'Fiction',
  'Electronic books',
  'Classics'])

In [91]:
df[['hardcover_title', 'hardcover_author', 'hardcover_rating', 'hardcover_ratings_count', 'release_year', 'users_count', 'genres']] = df.apply(
    lambda x: pd.Series(find_first_book_data(x['title'], x['author'])), axis=1)
df


,Unnamed: 0,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,...,weeks_on_list,isbns,buy_links,hardcover_title,hardcover_author,hardcover_rating,hardcover_ratings_count,release_year,users_count,genres
0,0,NaN,https://www.amazon.com/Where-Crawdads-Sing-Del...,NaN,0,Delia Owens,https://static01.nyt.com/bestsellers/images/97...,495,328,NaN,...,69,"[{'isbn10': '', 'isbn13': '9780735219090'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",NaN,None,NaN,NaN,NaN,NaN,None
1,1,NaN,https://www.amazon.com/Guardians-Novel-John-Gr...,NaN,0,John Grisham,https://static01.nyt.com/bestsellers/images/97...,481,330,NaN,...,12,"[{'isbn10': '', 'isbn13': '9780385544191'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",The Guardians,[John Grisham],4.054054,74.0,2019.0,165.0,"[Mystery, African Americans, Murder, Fiction, ..."
2,2,NaN,http://www.amazon.com/The-Last-Wish-Introducin...,NaN,0,Andrzej Sapkowski,https://static01.nyt.com/bestsellers/images/97...,495,330,NaN,...,2,"[{'isbn10': '', 'isbn13': '9780316055086'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",The Last Wish,"[Andrzej Sapkowski, Danusia Stok]",4.055982,1822.0,1975.0,3532.0,"[Fantasy, Adventure, Science Fiction & Fantasy..."
3,3,NaN,https://www.amazon.com/Wives-Novel-Tarryn-Fish...,NaN,0,Tarryn Fisher,https://static01.nyt.com/bestsellers/images/97...,494,330,NaN,...,1,"[{'isbn10': '', 'isbn13': '9781488054358'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",The Wives,[Tarryn Fisher],3.175676,148.0,2019.0,366.0,"[Thriller, Fiction, Abused women]"
4,4,NaN,https://www.amazon.com/Such-Fun-Age-Kiley-Reid...,NaN,0,Kiley Reid,https://static01.nyt.com/bestsellers/images/97...,495,328,https://www.nytimes.com/2019/12/31/books/revie...,...,1,"[{'isbn10': '', 'isbn13': '9780525541905'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",Such a Fun Age,[Kiley Reid],3.820866,508.0,2019.0,1260.0,"[Fiction, Contemporary, Romance, General, Afri..."
